In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sqlite3

print("Starting Advanced Analytics...")

# 1. Define your exact project root
project_root = Path(r"F:\bluestock_mf_capstone")

# 2. Build the exact paths
db_path = project_root / "data" / "db" / "bluestock_mf.db"
PROCESSED_DIR = project_root / "data" / "processed"

print(f"✅ Connected to REAL database at: {db_path}")
print(f"✅ Found Data folder at: {PROCESSED_DIR}")

# --- 3. Load Required Data ---
conn = sqlite3.connect(db_path)

# NAV Data
nav_path = PROCESSED_DIR / "clean_nav.csv"
df_nav = pd.read_csv(nav_path)
df_nav['date'] = pd.to_datetime(df_nav['nav_date'] if 'nav_date' in df_nav.columns else df_nav['date'])
df_nav = df_nav.sort_values(by=['amfi_code', 'date'])

# Fund Metadata (FIXED: Using 'risk_category AS risk_grade')
df_funds = pd.read_sql("SELECT amfi_code, scheme_name, category, risk_category AS risk_grade FROM dim_fund", conn)

# Transactions
df_tx = pd.read_sql("SELECT * FROM fact_transactions", conn)
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# Portfolio Holdings
df_port = pd.read_sql("SELECT * FROM fact_portfolio_holdings", conn)

print("✅ Data successfully loaded! Ready for Advanced Analytics.")

Starting Advanced Analytics...
✅ Connected to REAL database at: F:\bluestock_mf_capstone\data\db\bluestock_mf.db
✅ Found Data folder at: F:\bluestock_mf_capstone\data\processed
✅ Data successfully loaded! Ready for Advanced Analytics.


2: NAV Preprocessing (Handling Holidays)

In [5]:
# 1. Reindex to full business day calendar and ffill
all_dates = pd.date_range(start=df_nav['date'].min(), end=df_nav['date'].max(), freq='B')

def reindex_fund(group):
    group = group.set_index('date').reindex(all_dates).ffill()
    group['amfi_code'] = group['amfi_code'].ffill()
    group = group.reset_index().rename(columns={'index': 'date'})
    return group

# Apply reindexing to handle missing holiday/weekend data
df_nav_full = df_nav.groupby('amfi_code', group_keys=False).apply(reindex_fund)
df_nav_full['date'] = pd.to_datetime(df_nav_full['date'])

# 2. Calculate Daily Returns on the cleaned dataset
df_nav_full = df_nav_full.sort_values(by=['amfi_code', 'date'])
df_nav_full['daily_return'] = df_nav_full.groupby('amfi_code')['nav'].pct_change()

print("✅ Holidays Handled & Daily Returns Computed.")

KeyError: 'amfi_code'